<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_01a_transformer_seq2one_robustness_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_01a - SEQ2ONE - Transformer - Robustness testing**

# **BLOQUE DE EJECUCIÓN COMPLETO**


Del análisis de métricas concluímos que:



In [1]:
window_sizes = [90, 180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y

In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML con testing**


In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [22]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo Transformer — seq2one**

**Idea básica**

El **Transformer** es un modelo de aprendizaje profundo basado en el
mecanismo de **self-attention**, diseñado para modelar dependencias
temporales **sin recurrencia** y con procesamiento completamente
paralelo de la secuencia de entrada.

En el esquema **many-to-one**, el modelo recibe una **ventana temporal**
de múltiples pasos (many) y produce **un único valor escalar futuro**
(one), asociado al final de la ventana.

A diferencia del MLP, el Transformer **preserva explícitamente la
estructura temporal**, permitiendo que cada instante de la secuencia
atienda a cualquier otro instante según su relevancia para la
predicción final.

Formalmente, el modelo puede representarse como:

$$
\hat{y}_t = g\Big( \text{Pool}\big( \text{TransformerEncoder}(X_t) \big) \Big)
$$

donde:
- $X_t \in \mathbb{R}^{T \times F}$ es la ventana temporal (longitud $T$, $F$ features),
- $\text{TransformerEncoder}(\cdot)$ aplica capas de self-attention y feedforward,
- $\text{Pool}(\cdot)$ es una agregación temporal (último token, mean pooling, etc.),
- $g(\cdot)$ es una capa densa final que produce el target escalar.

---

**Regularización (Transformer)**

**Riesgo:** Alto, debido a la elevada capacidad del modelo.

El Transformer incorpora **regularización parcial de forma intrínseca**,
pero requiere control explícito:

- **Dropout (intrínseco):**
  - Aplicado en self-attention y capas feedforward.
- **Early stopping:**
  - Fundamental para evitar sobreajuste.
- **Dimensión del embedding controlada:**
  - Evita representaciones excesivamente complejas.
- **Número limitado de capas encoder:**
  - 1–3 capas en escenarios de datos financieros.
- **Weight decay (opcional):**
  - Refuerza la estabilidad del entrenamiento.

---

**Por qué el Transformer es relevante en este proyecto**

- Capacidad para capturar:
  - dependencias **de largo alcance**,
  - relaciones temporales no locales.
- Adecuado para:
  - ventanas largas (60–90 minutos),
  - múltiples indicadores técnicos simultáneos.
- Entrenamiento:
  - paralelo y estable,
  - más escalable que LSTM/GRU.

El Transformer representa el **primer modelo plenamente atencional**
del pipeline, sirviendo como referencia frente a arquitecturas
secuenciales (LSTM, GRU) y convolucionales (TCN).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas encoder: 1–2
- Dimensión del embedding: moderada (32–64)
- Número de cabezas de atención: 2–4
- Dropout: activado
- Pooling temporal: último token o mean pooling
- Early stopping: activado
- **Sin tuning exhaustivo** (optimización posterior)

El ajuste fino de profundidad, atención y regularización se aborda en
etapas posteriores del proyecto.


### **10.2. Imports y “seed” (base reproducible)**

In [23]:
# Paso 1: imports básicos + reproducibilidad (sin tqdm)
import os
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [24]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


### **10.3. TensorDataset + DataLoader (PyTorch)**

In [25]:
# Paso 3: TensorDataset y DataLoaders para H60 y H90 (sin tqdm)

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 1024,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para TRANSFORMER many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")

        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
        )

    return loaders

### **10.4. Definición de modelo Transformer (many-to-one)**

In [26]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        L = x.size(1)
        x = x + self.pe[:, :L, :]
        return self.dropout(x)

class TransformerManyToOne(nn.Module):
    def __init__(
        self,
        *,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" o "last"
    ):
        super().__init__()
        self.pooling = pooling

        self.in_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B, L, D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)
        z = self.in_proj(x)          # (B, L, D)
        z = self.pos_enc(z)          # (B, L, D)
        z = self.encoder(z)          # (B, L, D)

        if self.pooling == "last":
            pooled = z[:, -1, :]     # (B, D)
        else:
            pooled = z.mean(dim=1)   # (B, D)

        out = self.head(pooled)      # (B, 1)
        return out


In [27]:
# ============================================================
# 2) Modelo: factory (misma idea que antes)
# ============================================================
def make_transformer_model(*, n_features: int, device: torch.device) -> nn.Module:
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_ff=128,
        dropout=0.1,
        pooling="mean",
    ).to(device)
    return model


In [28]:
# ------------------------------------------------------------
# Modelos independientes (H60 y H90)
# IMPORTANTE: n_features debe coincidir con tu X 3D: (B, L, F)
# ------------------------------------------------------------
n_features = 36

In [29]:
# ------------------------------------------------------------
# Smoke test robusto (por horizonte)
# ------------------------------------------------------------
def smoke_test(model, loader, device, name: str):
    model.eval()
    xb, yb = next(iter(loader))
    xb = xb.to(device)
    yb = yb.to(device)

    with torch.no_grad():
        out = model(xb)

    print(f"[{name}] X:", tuple(xb.shape), xb.dtype)
    print(f"[{name}] y:", tuple(yb.shape), yb.dtype)
    print(f"[{name}] out:", tuple(out.shape), out.dtype)

    assert xb.ndim == 3, "X debe ser 3D: (B, L, F)"
    assert out.ndim == 2 and out.shape[1] == 1, "Salida debe ser (B, 1)"
    assert yb.ndim in (1, 2), "y debe ser (B,) o (B,1)"

### **10.5. Definición de loss, optimizer y funciones de train / eval (sin tqdm)**

In [30]:
# ============================================================
# loss, optimizer y funciones de entrenamiento / evaluación
# (preparado para múltiples targets / horizontes / window_size)
# ============================================================

import torch
import torch.nn as nn
from typing import Optional


# ------------------------------------------------------------
# Factory: Loss (regresión)
# ------------------------------------------------------------
def make_criterion() -> nn.Module:
    return nn.MSELoss()


# ------------------------------------------------------------
# Factory: Optimizer (uno por corrida/modelo)
# ------------------------------------------------------------
def make_optimizer(
    model: nn.Module,
    *,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
) -> torch.optim.Optimizer:
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


# ------------------------------------------------------------
# Función de entrenamiento (1 epoch)
# ------------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    *,
    clip_grad_norm: Optional[float] = None,
) -> float:
    model.train()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        loss.backward()

        if clip_grad_norm is not None and clip_grad_norm > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Función de evaluación (1 epoch)
# ------------------------------------------------------------
@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.eval()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        y_hat = model(xb)
        loss = criterion(y_hat, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Ejemplo de uso (por corrida / por horizonte)
# ------------------------------------------------------------
# criterion = make_criterion()
# optimizer_60 = make_optimizer(model_60, lr=1e-3, weight_decay=1e-4)
# train_loss = train_one_epoch(model_60, loaders_60["train"], optimizer_60, criterion, device, clip_grad_norm=1.0)
# valid_loss = eval_one_epoch(model_60, loaders_60["valid"], criterion, device)



### **10.6. Loop de entrenamiento completo con early stopping**



In [31]:
import math
import torch
import torch.nn as nn
from typing import Optional, Dict, Any


def fit_one_run(
    *,
    model: nn.Module,
    train_loader,
    valid_loader,
    device: torch.device,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    save_best: bool = True,
    best_path: Optional[str] = None,
    clip_grad_norm: Optional[float] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Entrena 1 modelo (1 target/horizonte/window_size) con early stopping en VALID.
    Retorna: history + best info. Deja el modelo restaurado al mejor estado si save_best=True.
    """
    criterion = make_criterion()
    optimizer = make_optimizer(model, lr=lr, weight_decay=weight_decay)

    best_val = math.inf
    best_epoch = -1
    patience_left = patience

    history = {"train_loss": [], "valid_loss": []}
    best_state = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device, clip_grad_norm=clip_grad_norm
        )
        val_loss = eval_one_epoch(
            model, valid_loader, criterion, device
        )

        history["train_loss"].append(float(train_loss))
        history["valid_loss"].append(float(val_loss))

        improved = (best_val - val_loss) > min_delta
        if improved:
            best_val = float(val_loss)
            best_epoch = epoch
            patience_left = patience

            if save_best:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                if best_path is not None:
                    torch.save(model.state_dict(), best_path)
        else:
            patience_left -= 1

        if verbose:
            print(
                f"epoch {epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"valid_loss={val_loss:.6f} | "
                f"patience_left={patience_left}"
            )

        if patience_left <= 0:
            if verbose:
                print(f"Early stopping: best_valid_loss={best_val:.6f} at epoch {best_epoch}")
            break

    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
        if verbose:
            print(f"Modelo restaurado: epoch {best_epoch} | best_valid_loss={best_val:.6f}")

    return {
        "best_valid_loss": best_val,
        "best_epoch": best_epoch,
        "epochs_ran": epoch,
        "history": history,
    }


### **10.7. Predicciones Transformer**


Función de predicción (seq2one) -> y_true, y_pred

In [32]:
import numpy as np
import torch

@torch.no_grad()
def predict_seq2one(
    model,
    loader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()

    y_true_list = []
    y_pred_list = []

    for xb, yb in loader:
        xb = xb.to(device)

        y_hat = model(xb).detach().cpu().numpy()   # (B,1) típico
        y_true = yb.detach().cpu().numpy()         # (B,1) o (B,)

        y_pred_list.append(y_hat)
        y_true_list.append(y_true)

    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all = np.concatenate(y_pred_list, axis=0)

    return y_true_all, y_pred_all

In [33]:
def get_metrics_torch_from_loaders(
    loaders: dict,
    model,
    *,
    device: torch.device,
    predict_loader_fn=predict_seq2one,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas valid/test para modelos seq2one usando DataLoaders.
    Espera loaders con keys: 'valid' y 'test'.
    predict_loader_fn debe devolver (y_true_all, y_pred_all).
    """

    # -------- VALID --------
    y_true_valid, y_pred_valid = predict_loader_fn(model, loaders["valid"], device=device)
    y_true_valid = np.asarray(y_true_valid).reshape(-1)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_true_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    y_true_test, y_pred_test = predict_loader_fn(model, loaders["test"], device=device)
    y_true_test = np.asarray(y_true_test).reshape(-1)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_true_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

### **11 Funciones de intregación**

### **11.1. Función `train_transformer`**

In [34]:
from typing import Any, Dict, Tuple
import torch
import torch.nn as nn

def train_transformer(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,

    # ---- NUEVO: seed opcional ----
    seed: int | None = None,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    use_scheduler: bool = False,   # por ahora lo ignoramos
    save_best: bool = True,
    best_path: str | None = None,
    verbose: bool = True,
) -> Tuple[nn.Module, Dict[str, Any], Dict, Dict]:
    """
    Entrena 1 Transformer (1 target/horizonte/window_size) y devuelve:
      (model, hist, metrics_valid, metrics_test)
    """

    # 0) ---- NUEVO: fijar semilla ANTES de crear el modelo ----
    if seed is not None:
        set_seed(int(seed))

    # 1) modelo
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    # 2) fit (early stopping en VALID)
    hist = fit_one_run(
        model=model,
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        min_delta=min_delta,
        save_best=save_best,
        best_path=best_path,
        clip_grad_norm=clip_grad_norm,
        verbose=verbose,
    )

    # 3) métricas VALID/TEST usando loaders
    metrics_valid, metrics_test = get_metrics_torch_from_loaders(
        {"valid": loaders["valid"], "test": loaders["test"]},
        model,
        device=device,
        predict_loader_fn=predict_seq2one,
        compute_r2=True,
    )

    return model, hist, metrics_valid, metrics_test

### **11.2. Función `run_transformer`**

In [35]:
# ============================================================
# run_transformer (con loop por seeds) + run_transformer_incremental (skip por seeds)
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Iterable, Any, Dict, Tuple

import gc
import time
import pandas as pd
import torch


def _ts() -> str:
    return time.strftime("%H:%M:%S")


# ------------------------------------------------------------
# RUN 1 window_size (L) con robustez por seeds
# ------------------------------------------------------------
def run_transformer(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- robustez ----
    seeds: int | list[int] = 42,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,  # (por ahora ignorado en train_transformer)
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta Transformer seq2one para:
      - target fijo: delta_60
      - window_size fijo: L
    Repitiendo entrenamiento para múltiples seeds (robustez).

    Retorna un DataFrame con filas valid/test por seed.
    """
    L = int(window_size)

    # normalizar seeds
    if isinstance(seeds, int):
        seeds_list = [int(seeds)]
    else:
        seeds_list = [int(s) for s in seeds]

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] TRANSFORMER | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| d_model={d_model} | head={nhead} | layers={num_layers} | ff={dim_ff} | do={dropout} "
            f"| wd={weight_decay} | seeds={seeds_list}"
        )
        print("=" * 80)

    target = "delta_60"
    rows = []
    t_global = time.perf_counter()

    # -------------------------
    # BUILD BUNDLE (3D) (no depende de seed)
    # -------------------------
    bundle = None
    try:
        if verbose:
            print(f"\n[{_ts()}] [BUILD] Creando bundle (flatten_X=False) | target='{target}' | L{L} ...")
        t0 = time.perf_counter()

        (bundle,) = create_bundles(
            window_size=L,
            targets=[target],
            windows_paths=windows_paths,
            scalers_paths=scalers_paths,
            flatten_X=False,  # Transformer necesita 3D
        )

        if verbose:
            dt = time.perf_counter() - t0
            try:
                xshape = bundle["train"]["X"].shape
                yshape = bundle["train"]["y"].shape
                print(f"[{_ts()}] [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
            except Exception:
                print(f"[{_ts()}] [BUILD] OK | dt={dt:.2f}s")

        # ==========================================================
        # LOOP POR SEED (robustez)
        # ==========================================================
        for j, seed in enumerate(seeds_list, start=1):
            loaders = None
            model = None
            hist = None
            metrics_valid = None
            metrics_test = None

            try:
                if verbose:
                    print(f"\n[{_ts()}] [SEED] ({j}/{len(seeds_list)}) seed={seed}")

                # -------------------------
                # SEED (antes de loaders/modelo)
                # -------------------------
                set_seed(seed)

                # -------------------------
                # LOADERS (train/valid/test)
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
                t0 = time.perf_counter()

                loaders = make_loaders_from_bundle_3d(
                    bundle,
                    seq_len=L,
                    n_features=n_features,
                    batch_size=batch_size_train,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    try:
                        ntr = len(loaders["train"].dataset)
                        nva = len(loaders["valid"].dataset)
                        nte = len(loaders["test"].dataset)
                        print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                    except Exception:
                        print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

                # -------------------------
                # TRAIN
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
                t0 = time.perf_counter()

                model, hist, metrics_valid, metrics_test = train_transformer(
                    loaders,
                    n_features=n_features,
                    device=device,
                    seed=seed,  # <- para inicialización del modelo dentro de train_transformer
                    d_model=d_model,
                    nhead=nhead,
                    num_layers=num_layers,
                    dim_ff=dim_ff,
                    dropout=dropout,
                    pooling=pooling,
                    lr=lr,
                    weight_decay=weight_decay,
                    max_epochs=max_epochs,
                    patience=patience,
                    clip_grad_norm=clip_grad_norm,
                    use_scheduler=use_scheduler,
                    verbose=verbose,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

                # -------------------------
                # PRED LOADERS (valid/test) con batch grande
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [PRED] Preparando loaders valid/test (batch_size_pred={batch_size_pred}) ...")
                t0 = time.perf_counter()

                pred_bundle = {
                    "valid": bundle["valid"],
                    "test": bundle["test"],
                    # dummy para cumplir interfaz sin usar train
                    "train": {"X": bundle["valid"]["X"][:1], "y": bundle["valid"]["y"][:1]},
                }
                pred_loaders = make_loaders_from_bundle_3d(
                    pred_bundle,
                    seq_len=L,
                    n_features=n_features,
                    batch_size=batch_size_pred,
                    num_workers=0,  # pred más estable con 0
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [PRED] Loaders OK | dt={dt:.2f}s")

                # -------------------------
                # METRICS (valid/test)
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [METRICS] Calculando métricas (valid/test) ...")
                t0 = time.perf_counter()

                metrics_valid, metrics_test = get_metrics_torch_from_loaders(
                    {"valid": pred_loaders["valid"], "test": pred_loaders["test"]},
                    model,
                    device=device,
                    predict_loader_fn=predict_seq2one,
                    compute_r2=True,
                )

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

                # -------------------------
                # DF APPEND (con seed)
                # -------------------------
                if verbose:
                    print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
                t0 = time.perf_counter()

                df_v = metrics_to_df(
                    metrics_valid,
                    model="transformer",
                    split="valid",
                    horizon=bundle["horizon"],
                    window_size=bundle["window_size"],
                    target=bundle["target"],
                )

                df_t = metrics_to_df(
                    metrics_test,
                    model="transformer",
                    split="test",
                    horizon=bundle["horizon"],
                    window_size=bundle["window_size"],
                    target=bundle["target"],
                )

                for df_ in (df_v, df_t):
                    # robustez
                    df_["seed"] = seed

                    # hparams
                    df_["d_model"] = d_model
                    df_["nhead"] = nhead
                    df_["num_layers"] = num_layers
                    df_["dim_ff"] = dim_ff
                    df_["dropout"] = dropout
                    df_["pooling"] = pooling
                    df_["lr"] = lr
                    df_["weight_decay"] = weight_decay

                    # training info (fit_one_run devuelve best_valid_loss)
                    if isinstance(hist, dict):
                        df_["best_valid_mse"] = hist.get("best_valid_loss")
                        df_["best_epoch"] = hist.get("best_epoch")
                        df_["epochs_ran"] = hist.get("epochs_ran")

                rows.append(df_v)
                rows.append(df_t)

                if verbose:
                    dt = time.perf_counter() - t0
                    print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            finally:
                if verbose:
                    print(f"[{_ts()}]   [CLEAN] Liberando objetos seed={seed} ...")

                loaders = model = hist = metrics_valid = metrics_test = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    finally:
        if verbose:
            print(f"[{_ts()}] [CLEAN] Liberando bundle ...")
        bundle = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_transformer_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_transformer_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(
            df_transformer_metrics[["window_size", "target", "seed", "split", "horizon_min", "model"]]
            .drop_duplicates()
            .to_string(index=False)
        )

    return df_transformer_metrics


### **11.4. Función `run_transformer_incremental`**

In [36]:
from pathlib import Path
import pandas as pd
import gc
import torch

def run_transformer_incremental(
    *,
    window_sizes: list[int],

    # ---- robustez ----
    seeds: int | list[int] = 42,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = True,

    # ---- data ----
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- persistencia ----
    name: str = "transformer",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Incremental para Transformer (target fijo delta_60), guardando progreso POR SEED:
      - Para cada L en window_sizes:
          - Calcula seeds faltantes (requiere valid y test por seed)
          - Ejecuta run_transformer(L, seeds=[seed]) por cada seed faltante
          - Guarda df_hist inmediatamente tras cada seed
    """

    # normalizar seeds
    if isinstance(seeds, int):
        seeds_list = [int(seeds)]
    else:
        seeds_list = [int(s) for s in seeds]
    expected_seeds = set(seeds_list)

    metrics_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_testing")
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    # cargar histórico
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    expected_target = "delta_60"
    expected_splits = {"valid", "test"}

    for L in window_sizes:
        L = int(L)

        # --------- determinar seeds faltantes para este L ----------
        missing_seeds = set(expected_seeds)

        if (not df_hist.empty) and {"seed", "split", "target", "window_size", "model"}.issubset(df_hist.columns):
            dfL = df_hist[
                (df_hist["model"] == name) &
                (df_hist["window_size"] == L) &
                (df_hist["target"] == expected_target) &
                (df_hist["seed"].isin(expected_seeds)) &
                (df_hist["split"].isin(expected_splits))
            ]

            # Para cada seed: splits presentes
            splits_by_seed = dfL.groupby("seed")["split"].apply(set).to_dict()
            complete_seeds = {s for s, sp in splits_by_seed.items() if expected_splits.issubset(sp)}

            missing_seeds = set(expected_seeds) - complete_seeds

        if not missing_seeds:
            if verbose:
                print(f"[SKIP] {name} L={L} {expected_target} ya existe completo para seeds={sorted(expected_seeds)}")
            continue

        if verbose:
            print(f"[RUN] {name} L={L} faltan seeds={sorted(missing_seeds)}")

        # --------- ejecutar y GUARDAR por cada seed ----------
        for seed in sorted(missing_seeds):
            if verbose:
                print(f"[RUN] {name} L={L} -> seed={seed} (guardado inmediato)")

            df_seed = None
            try:
                df_seed = run_transformer(
                    window_size=L,
                    seeds=[seed],  # <- 1 seed por corrida
                    n_features=n_features,
                    batch_size_train=batch_size_train,
                    batch_size_pred=batch_size_pred,
                    d_model=d_model,
                    nhead=nhead,
                    num_layers=num_layers,
                    dim_ff=dim_ff,
                    dropout=dropout,
                    pooling=pooling,
                    lr=lr,
                    weight_decay=weight_decay,
                    max_epochs=max_epochs,
                    patience=patience,
                    clip_grad_norm=clip_grad_norm,
                    use_scheduler=use_scheduler,
                    verbose=verbose,
                )

                # asegurar model name
                df_seed["model"] = name

                # anexar a histórico en memoria (pequeño: 2 filas por seed)
                if df_hist.empty:
                    df_hist = df_seed.copy()
                else:
                    df_hist = pd.concat([df_hist, df_seed], ignore_index=True)

                # GUARDAR inmediatamente (checkpoint)
                save_seq2one_metrics(df_hist, name=name)

            finally:
                # liberar objetos pesados y cache GPU
                df_seed = None
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # (opcional) recargar df_hist desde disco para minimizar memoria y garantizar consistencia
            # con muchos runs. Coméntalo si no lo necesitas.
            if metrics_path.exists():
                df_hist = pd.read_parquet(metrics_path)

    return (
        df_hist.sort_values(["window_size", "target", "seed", "split", "horizon_min", "model"])
              .reset_index(drop=True)
    )

## **12. Aplicación**

In [37]:
seeds = [
    1, 7, 42, 123, 999,
    2024, 31415, 27182, 8080, 777,
    5555, 8888, 1001, 2025, 9090,
    3333, 4444, 6666, 1212, 2121
]

In [ ]:
df_transformer_all_sizes = run_transformer_incremental(
    window_sizes=[90, 180],

    # robustez
    seeds = seeds,

    # hiperparámetros
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_ff=128,
    dropout=0.1,
    pooling="mean",

    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=30,
    patience=5,
    clip_grad_norm=1.0,
    use_scheduler=True,

    name="transformer",
    verbose=True,
)

[RUN] transformer L=90 faltan seeds=[1, 7, 42, 123, 777, 999, 1001, 1212, 2024, 2025, 2121, 3333, 4444, 5555, 6666, 8080, 8888, 9090, 27182, 31415]
[RUN] transformer L=90 -> seed=1 (guardado inmediato)

[14:12:02] TRANSFORMER | SEQ2ONE | WINDOW_SIZE=L90 | (L,F)=(90,36) | d_model=64 | head=4 | layers=2 | ff=128 | do=0.1 | wd=0.0001 | seeds=[1]

[14:12:02] [BUILD] Creando bundle (flatten_X=False) | target='delta_60' | L90 ...
H60 Train: (409512, 90, 36) (409512,)
H60 Valid: (87688, 90, 36) (87688,)
H60 Test : (88140, 90, 36) (88140,)
Scaler H60: StandardScaler
[14:12:16] [BUILD] OK | train X=(409512, 90, 36) y=(409512,) | dt=13.93s

[14:12:16] [SEED] (1/1) seed=1
[14:12:16]   [LOADERS] Creando DataLoaders (3D) ...
[14:12:23]   [LOADERS] OK | n(train/valid/test)=(409512/87688/88140) | dt=7.09s
[14:12:23]   [TRAIN] Iniciando entrenamiento ...


/tmp/ipykernel_3063/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=3091.215968 | valid_loss=2866.164059 | patience_left=5
epoch 02 | train_loss=3039.081909 | valid_loss=2855.119398 | patience_left=5
epoch 03 | train_loss=2943.682619 | valid_loss=3000.741959 | patience_left=4
epoch 04 | train_loss=2806.576772 | valid_loss=3123.053885 | patience_left=3
epoch 05 | train_loss=2670.139094 | valid_loss=3321.850543 | patience_left=2
epoch 06 | train_loss=2549.160852 | valid_loss=3569.234654 | patience_left=1
epoch 07 | train_loss=2448.297916 | valid_loss=3706.312670 | patience_left=0
Early stopping: best_valid_loss=2855.119398 at epoch 2
Modelo restaurado: epoch 2 | best_valid_loss=2855.119398
[14:17:06]   [TRAIN] FIN entrenamiento | dt=282.86s
[14:17:06]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[14:17:08]   [PRED] Loaders OK | dt=1.93s
[14:17:08]   [METRICS] Calculando métricas (valid/test) ...
[14:17:15]   [METRICS] OK (valid/test) | dt=7.36s
[14:17:15]   [DF] Agregando filas a la tabla ...
[14:17:15]   [DF] 

/tmp/ipykernel_3063/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=3089.162218 | valid_loss=2813.490094 | patience_left=5
epoch 02 | train_loss=3036.526860 | valid_loss=2855.167100 | patience_left=4
epoch 03 | train_loss=2949.829693 | valid_loss=2991.849257 | patience_left=3
epoch 04 | train_loss=2826.399981 | valid_loss=3503.830069 | patience_left=2
epoch 05 | train_loss=2698.606008 | valid_loss=3577.407819 | patience_left=1
epoch 06 | train_loss=2573.145705 | valid_loss=3582.257776 | patience_left=0
Early stopping: best_valid_loss=2813.490094 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=2813.490094
[14:21:35]   [TRAIN] FIN entrenamiento | dt=244.59s
[14:21:35]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[14:21:37]   [PRED] Loaders OK | dt=2.00s
[14:21:37]   [METRICS] Calculando métricas (valid/test) ...
[14:21:44]   [METRICS] OK (valid/test) | dt=7.24s
[14:21:44]   [DF] Agregando filas a la tabla ...
[14:21:44]   [DF] OK | dt=0.01s
[14:21:44]   [CLEAN] Liberando objetos seed=7 ...
[14:21:44] [C

...

epoch 01 | train_loss=3089.213796 | valid_loss=2795.734254 | patience_left=5
epoch 02 | train_loss=3045.169892 | valid_loss=2848.077580 | patience_left=4
epoch 03 | train_loss=2966.144885 | valid_loss=3057.885026 | patience_left=3
epoch 04 | train_loss=2863.297257 | valid_loss=2995.265434 | patience_left=2
epoch 05 | train_loss=2757.697996 | valid_loss=3186.704776 | patience_left=1
epoch 06 | train_loss=2638.818240 | valid_loss=3278.200510 | patience_left=0
Early stopping: best_valid_loss=2795.734254 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=2795.734254
[15:42:16]   [TRAIN] FIN entrenamiento | dt=244.73s
[15:42:16]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[15:42:18]   [PRED] Loaders OK | dt=1.91s
[15:42:18]   [METRICS] Calculando métricas (valid/test) ...
[15:42:25]   [METRICS] OK (valid/test) | dt=7.19s
[15:42:25]   [DF] Agregando filas a la tabla ...
[15:42:25]   [DF] OK | dt=0.01s
[15:42:25]   [CLEAN] Liberando objetos seed=31415 ...
[15:42:25

/tmp/ipykernel_3063/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=3346.033358 | valid_loss=3002.553710 | patience_left=5
epoch 02 | train_loss=3309.073937 | valid_loss=3220.646839 | patience_left=4
epoch 03 | train_loss=3220.288237 | valid_loss=3946.456640 | patience_left=3
epoch 04 | train_loss=3067.962300 | valid_loss=4097.481918 | patience_left=2
epoch 05 | train_loss=2907.136731 | valid_loss=5673.700240 | patience_left=1
epoch 06 | train_loss=2765.087804 | valid_loss=7414.322381 | patience_left=0
Early stopping: best_valid_loss=3002.553710 at epoch 1
Modelo restaurado: epoch 1 | best_valid_loss=3002.553710
[15:50:04]   [TRAIN] FIN entrenamiento | dt=432.35s
[15:50:04]   [PRED] Preparando loaders valid/test (batch_size_pred=32768) ...
[15:50:07]   [PRED] Loaders OK | dt=3.09s
[15:50:07]   [METRICS] Calculando métricas (valid/test) ...
[15:50:18]   [METRICS] OK (valid/test) | dt=11.17s
[15:50:18]   [DF] Agregando filas a la tabla ...
[15:50:18]   [DF] OK | dt=0.01s
[15:50:18]   [CLEAN] Liberando objetos seed=1 ...
[15:50:19] [

/tmp/ipykernel_3063/3848978488.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


epoch 01 | train_loss=3346.348816 | valid_loss=2986.010726 | patience_left=5
epoch 02 | train_loss=3298.414767 | valid_loss=3257.701619 | patience_left=4
epoch 03 | train_loss=3189.406789 | valid_loss=4131.141774 | patience_left=3
epoch 04 | train_loss=3027.858115 | valid_loss=4291.167355 | patience_left=2
epoch 05 | train_loss=2864.012120 | valid_loss=7989.234816 | patience_left=1


In [ ]:
df_transformer_all_sizes

In [ ]:
save_seq2one_metrics(
    df_transformer_all_sizes,
    name="transformer"
)

## **13. Análisis**

In [ ]:
df_transformer_all_sizes

### **13.1. Análisis descriptivo comparativo por tamaño de ventana (L=90 vs L=180)**

El primer análisis (mínimo y más informativo) es un resumen estadístico por window_size y split para comparar promedio y dispersión (robustez) de las métricas.

In [ ]:
metrics = ["MAE", "RMSE", "R2", "DA"]

summary = (
    df_transformer_all_sizes.groupby(["window_size", "split"])[metrics]
      .agg(["mean", "std", "min", "max", "count"])
      .round(6)
)

summary

**Análisis descriptivo comparativo por tamaño de ventana (20 seeds)**

**1. Desempeño promedio en TEST**

- MAE  
  - L=90 → 38.36  
  - L=180 → 35.39  
  Mejora de aproximadamente **2.96 puntos**.

- RMSE  
  - L=90 → 66.63  
  - L=180 → 64.90  
  Mejora consistente (~1.74 puntos).

- R²  
  - L=90 → 0.3386  
  - L=180 → 0.3819  
  Incremento de **+0.043**.

- Directional Accuracy (DA)  
  - L=90 → 0.7217  
  - L=180 → 0.7465  
  Mejora clara (~+0.025).

- Conclusión TEST:
  - La ventana **L=180 domina a L=90** en MAE, RMSE, R² y DA.  
  - La mejora es relevante tanto en magnitud del error como en varianza explicada y precisión direccional.


**2. Robustez en TEST (desviación estándar)**

- R² (std)  
  - L=90 → 0.0505  
  - L=180 → 0.0557  

- MAE (std)  
  - L=90 → 1.998  
  - L=180 → 2.358  

La dispersión es baja en ambos casos.  
L=180 presenta levemente mayor variabilidad en TEST, pero dentro de un rango estable y sin evidencia de inestabilidad estructural.

**3. Desempeño promedio en VALID**

- MAE  
  - L=90 → 22.22  
  - L=180 → 20.67  

- RMSE  
  - L=90 → 35.60  
  - L=180 → 33.85  

- R²  
  - L=90 → 0.4660  
  - L=180 → 0.4953  
  Incremento de **+0.029**.

- DA  
  - L=90 → 0.7424  
  - L=180 → 0.7596  

- Conclusión VALID:
  - La ventana **L=180 mejora consistentemente** todas las métricas en validación.  
  - La mejora en R² y en error absoluto confirma que el mayor contexto aporta señal útil.

**4. Gap VALID–TEST (generalización)**

- L=90  
  - VALID R² = 0.4660  
  - TEST R² = 0.3386  
  - Gap ≈ 0.127  

- L=180  
  - VALID R² = 0.4953  
  - TEST R² = 0.3819  
  - Gap ≈ 0.113  

El gap se reduce ligeramente al usar ventana más larga.  
No hay evidencia de incremento de sobreajuste con L=180.


**5. Conclusión general**

Con 20 seeds:

- L=180 mejora de forma clara el desempeño promedio en TEST.  
- Mejora también en VALID en todas las métricas.  
- Mantiene estabilidad entre inicializaciones.  
- Reduce ligeramente el gap VALID–TEST.  
- No incrementa el overfitting.  
- Aporta señal útil adicional respecto a L=90.

En el Transformer, al igual que en MLP, **la ventana L=180 domina estructuralmente a L=90**.

### **13.2. Test estadístico formal**

Queremos responder:

> ¿La mejora en R² en TEST al pasar de L=90 a L=180 es estadísticamente significativa?

Como usaste las mismas seeds, el test correcto es:
- Paired t-test (muestras dependientes)
- Alternativamente Wilcoxon (no paramétrico)

In [ ]:
df=df_transformer_all_sizes.copy()

**1. Preparar los vectores de R² (TEST)**

In [ ]:
import numpy as np

# Filtrar solo TEST
df_test = df[df["split"] == "test"]

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .values
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .values
)

r2_90, r2_180

**2. Paired t-test**

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_value = ttest_rel(r2_180, r2_90)
print("Paired t-test:")
t_stat, p_value

**3. Wilcoxon (más robusto con pocas muestras)**

In [ ]:
from scipy.stats import wilcoxon

w_stat, p_wilcoxon = wilcoxon(r2_180, r2_90)
print("Wilcoxon:")
w_stat, p_wilcoxon

**Análisis estadístico formal: L=90 vs L=180 (R² en TEST, 20 seeds)**


**1. Paired t-test**

- Estadístico t = 2.5484  
- p-value = 0.0196  

Interpretación:  
- El p-value es menor que 0.05.  
- Se rechaza la hipótesis nula de igualdad de medias al 5% de significancia.

Existe una diferencia estadísticamente significativa entre L=90 y L=180 en R².

**2. Test no paramétrico de Wilcoxon**

- Estadístico W = 45.0  
- p-value = 0.0240  

Interpretación:  
- También menor que 0.05.  
- La diferencia es significativa incluso sin asumir normalidad en las diferencias.


**3. Conclusión estadística**

Ambos tests confirman que:

- La mejora de R² al pasar de L=90 a L=180  
- No es producto del azar de inicialización  
- Es consistente a través de las 20 seeds  

La magnitud del estadístico t (≈ 2.55) indica que la diferencia es estadísticamente significativa, aunque el efecto es más moderado que en el caso del MLP.

Por lo tanto, manteniendo el mismo modelo e hiperparámetros, la ventana de 180 muestras es estadísticamente superior a 90 para el target delta_60 en el caso del Transformer.

### **13.3. Evaluación del tamaño del efecto (Effect Size – Cohen’s d)**

Hasta ahora demostramos que la diferencia entre L=90 y L=180 es estadísticamente significativa (p < 0.05).

Sin embargo, la significancia estadística solo responde a la pregunta:

- ¿La diferencia existe?

No responde a la pregunta más importante desde el punto de vista práctico:

- ¿La diferencia es grande o relevante?

El tamaño del efecto (Cohen’s d) mide la magnitud real de la diferencia entre ambos modelos en relación con la variabilidad entre seeds.

En términos simples:

- Si d es pequeño → la diferencia existe, pero su impacto es débil.
- Si d es moderado → la mejora es relevante.
- Si d es grande → el cambio de ventana tiene un impacto fuerte y consistente.

Objetivo en este análisis:

Cuantificar qué tan importante es la mejora en R² al pasar de ventana 90 a ventana 180, más allá de que sea estadísticamente significativa.

In [ ]:
import numpy as np

# 1) Tomar R² en TEST y alinear por seed
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .sort_values("seed")["R2"]
    .to_numpy()
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .sort_values("seed")["R2"]
    .to_numpy()
)

# 2) Diferencias pareadas (L180 - L90)
diff = r2_180 - r2_90

# 3) Cohen's d para muestras pareadas (dz): media(diff) / std(diff)
d_z = diff.mean() / diff.std(ddof=1)

# 4) Resumen útil
out = {
    "n": int(diff.size),
    "mean_R2_90": float(r2_90.mean()),
    "mean_R2_180": float(r2_180.mean()),
    "mean_diff": float(diff.mean()),
    "std_diff": float(diff.std(ddof=1)),
    "cohens_dz": float(d_z),
}

out

**Tamaño del efecto (Cohen’s d – muestras pareadas)**


Resultados:

- n = 20 seeds
- R² medio L=90  = 0.3386
- R² medio L=180 = 0.3819
- Diferencia media = +0.0433
- Desvío estándar de las diferencias = 0.0760
- Cohen’s d (dz) = 0.57

**Interpretación del tamaño del efecto:**

Reglas generales para Cohen’s d:

- 0.2 → efecto pequeño
- 0.5 → efecto moderado
- 0.8 → efecto grande

En este caso:

d = 0.57

Esto indica un efecto moderado.

Conclusión práctica:

- La mejora al pasar de L=90 a L=180 no solo es estadísticamente significativa.
- La magnitud del efecto es moderada.
- La mejora es relevante en términos prácticos.
- El incremento de memoria temporal aporta información útil y consistente.

Interpretación global (significancia + efecto):

- Existe diferencia real (p < 0.05).
- La diferencia no es trivial.
- El Transformer está explotando contexto adicional de forma estructural.


### **13.4. Análisis de la distribución de las diferencias por seed**

Hasta ahora sabemos que:

- L=180 es mejor en promedio.
- La diferencia es estadísticamente significativa.
- El tamaño del efecto es moderado.

Pero todavía no sabemos algo clave:

> ¿La mejora ocurre de manera consistente en casi todas las seeds,
o está siendo impulsada por unas pocas inicializaciones muy favorables?

En este punto buscamos:
- Analizar la distribución de las diferencias individuales (R²_180 − R²_90).
- Ver cuántas seeds realmente mejoran.
- Evaluar si la mejora es homogénea o depende de casos extremos.

**Objetivo concreto:**

Confirmar que la superioridad de L=180 es estructural y no producto de unas pocas semillas atípicas.

In [ ]:
import pandas as pd
import numpy as np

# --- Recalcular por claridad ---
df_test = df[df["split"] == "test"].copy()

r2_90 = (
    df_test[df_test["window_size"] == 90]
    .set_index("seed")["R2"]
)

r2_180 = (
    df_test[df_test["window_size"] == 180]
    .set_index("seed")["R2"]
)

df_diff = pd.DataFrame({
    "R2_90": r2_90,
    "R2_180": r2_180,
})

df_diff["diff"] = df_diff["R2_180"] - df_diff["R2_90"]

# ============================================================
# 1) Tabla ordenada
# ============================================================

df_sorted = (
    df_diff
    .sort_values("diff", ascending=False)
    .round(6)
)

print("\n=== DIFERENCIAS POR SEED (ordenado por mejora) ===")
display(df_sorted)

# ============================================================
# 2) Resumen estadístico
# ============================================================

summary = (
    df_diff["diff"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .to_frame()
    .T
    .round(6)
)

print("\n=== RESUMEN DE DIFERENCIAS (L180 - L90) ===")
display(summary)

# ============================================================
# 3) Conteo de mejoras
# ============================================================

n = len(df_diff)
n_pos = (df_diff["diff"] > 0).sum()
n_neg = (df_diff["diff"] < 0).sum()

improvement = pd.DataFrame([{
    "n_total": n,
    "n_mejora": n_pos,
    "n_empeora": n_neg,
    "pct_mejora": round(n_pos / n, 4),
    "pct_empeora": round(n_neg / n, 4),
}])

print("\n=== CONSISTENCIA DE LA MEJORA ===")
display(improvement)

**Análisis de la distribución de las diferencias por seed (MLP)**

**1. Consistencia de la mejora**

- Total de seeds: 20
- Seeds donde L=180 mejora a L=90: 15
- Seeds donde L=180 empeora respecto a L=90: 5
- Proporción de mejora: 75%
- Proporción de deterioro: 25%

Interpretación:
La mejora no depende de un caso aislado.  
En 3 de cada 4 inicializaciones, la ventana 180 supera a la 90.

**2. Magnitud de las diferencias**

- Diferencia media: +0.0433
- Mediana: +0.0490
- Desvío estándar: 0.0760
- Mejor mejora observada: +0.1541
- Peor deterioro observado: −0.1200

Observación clave:
La mediana es positiva y cercana a la media, lo que indica que la mejora no está impulsada por un único outlier extremo.

**3. Estructura de los casos negativos**

Existen 5 seeds donde L=180 empeora.

Sin embargo:

- Las mejoras máximas (+0.15, +0.14, +0.13) son mayores que la mayoría de los deterioros.
- Solo una seed presenta un deterioro fuerte (-0.12).
- La mayoría de los deterioros son moderados.

**4. Conclusión estructural**

La superioridad de L=180:

- No es producto de una única seed favorable.
- No depende de casos extremos aislados.
- Es consistente en la mayoría de inicializaciones.
- Tiene distribución razonablemente balanceada.

La mejora es estructural, no accidental.

### **13.5. Conclusión final del análisis comparativo L=90 vs L=180 (Transformer – delta_60)**

El análisis realizado es metodológicamente completo y robusto:

- Se evaluaron 20 seeds independientes.
- Se realizó comparación descriptiva por ventana y split.
- Se aplicaron tests estadísticos formales (t-test pareado y Wilcoxon).
- Se calculó tamaño del efecto (Cohen’s d).
- Se analizó la distribución de diferencias por seed.
- Se evaluó la consistencia de la mejora (75% de las seeds mejoran).
- Se revisó el gap de generalización (valid vs test).

La conclusión no se basa en una observación puntual, sino en evidencia estadística y estructural consistente.

Conclusión técnica:

- La ventana L=180 es superior a L=90 para el target `delta_60`.
- La mejora es estadísticamente significativa (p < 0.05).
- El tamaño del efecto es moderado (d ≈ 0.57).
- La mejora es consistente en la mayoría de inicializaciones.
- No se observa incremento del overfitting.
- La mejora no depende de una única seed extrema.

En consecuencia, el análisis puede considerarse formalmente cerrado y la ventana L=180 puede adoptarse como configuración preferente para el Transformer en este target.
